# Step 5c — Replication feasibility in the Siletti et al. 2023 human brain atlas

**Project:** pd-da-aging · **Why:** GSE157783 has too few dopamine neurons and GSE184950 cannot resolve the subtypes (see docs/ANALYSIS_PLAN.md). This atlas sequenced healthy substantia nigra deeply and its authors matched its dopamine neurons to Kamath's subtypes.

Run the cells **one at a time** and stop at each 🛑. Results go to *My Drive › pd-da-aging_step5c*. Runtime: CPU.

## Cell 1 — Connect Google Drive and install tools
Click **Allow**. The red pip warning about pandas is harmless.

In [ ]:
# Connect Google Drive and install tools (1-2 minutes)
from google.colab import drive
drive.mount("/content/drive")
!pip install -q scanpy anndata


## Cell 2 — Find the Siletti et al. 2023 atlas on CELLxGENE
Lists the atlas's midbrain datasets with their sizes. Nothing large is downloaded yet.
### 🛑 Send Claude a screenshot of the list.

In [ ]:
# Find the Siletti et al. 2023 human brain atlas on CELLxGENE and list its midbrain datasets (seconds)
import os, re, json, datetime, sys, platform, urllib.request, shutil
import numpy as np, pandas as pd, anndata as ad
from scipy import sparse

DRIVE = "/content/drive/MyDrive"
OUT = f"{DRIVE}/pd-da-aging_step5c"
for sub in ["", "/da_candidates", "/tables"]:
    os.makedirs(OUT + sub, exist_ok=True)
API = "https://api.cellxgene.cziscience.com/curation/v1"
def get_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())

cols = get_json(f"{API}/collections")
hits = [c for c in cols if re.search(r"human brain cell atlas|across the adult human brain", c.get("name", ""), re.I)
        or "add7046" in str(c.get("doi", ""))]
for c in hits:
    print(f"Collection: {c['name']}  |  id {c['collection_id']}  |  doi {c.get('doi')}")
rows = []
for c in hits:
    full = get_json(f"{API}/collections/{c['collection_id']}")
    for d in full.get("datasets", []):
        h5 = [a for a in d.get("assets", []) if str(a.get("filetype", "")).upper() == "H5AD"]
        rows.append({"collection": c["name"], "title": d.get("title"), "cells": d.get("cell_count"),
                     "dataset_id": d.get("dataset_id"),
                     "h5ad_url": h5[0]["url"] if h5 else None,
                     "h5ad_MB": round(h5[0].get("filesize", 0) / 1e6) if h5 else None})
datasets = pd.DataFrame(rows)
datasets.to_csv(f"{OUT}/tables/cellxgene_datasets.csv", index=False)
mid = datasets[datasets["title"].str.contains(r"substantia|nigra|\bSN\b|splatter|midbrain", case=False, regex=True, na=False)]
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90)
print(f"\n{len(datasets)} datasets in total; candidates for dopamine neurons:")
print(mid[["title", "cells", "h5ad_MB"]].to_string())


## Cell 3 — Download the substantia nigra dissection(s) and count dopamine neurons per donor
About 10–30 minutes. Keeps only the dopamine-neuron candidates in Drive.
### 🛑 Send Claude a screenshot of the whole output.

In [ ]:
# Download the substantia nigra dissection(s), keep the dopamine-neuron candidates (10-30 minutes)
CHOSEN = mid[mid["title"].str.contains(r"substantia|nigra", case=False, regex=True)]
print(f"Using {len(CHOSEN)} dataset(s): {list(CHOSEN['title'])}")
summary = []
for _, r in CHOSEN.iterrows():
    local = f"/content/{r['dataset_id']}.h5ad"
    if not os.path.exists(local):
        req = urllib.request.Request(r["h5ad_url"], headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as resp, open(local, "wb") as fh:
            shutil.copyfileobj(resp, fh, length=16 * 1024 * 1024)
    a = ad.read_h5ad(local)
    print(f"\n{r['title']}: {a.n_obs:,} nuclei; obs columns: {list(a.obs.columns)[:40]}")
    # CELLxGENE stores raw counts in .raw when .X is normalised; genes are Ensembl IDs with symbols in feature_name
    X = a.raw.X if a.raw is not None else a.X
    var = a.raw.var if a.raw is not None else a.var
    sym = var["feature_name"].astype(str).to_numpy() if "feature_name" in var else np.asarray(var.index, dtype=str)
    X = X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(X)
    is_int = np.allclose(X[:200].data, np.round(X[:200].data))
    print(f"Counts look like raw integers: {is_int}")
    col = {g: (np.flatnonzero(sym == g)[0] if (sym == g).any() else None) for g in ["TH", "SLC6A3", "SLC18A2", "SOX6", "CALB1", "AGTR1"]}
    gx = {g: (np.asarray(X[:, j].todense()).ravel() if j is not None else np.zeros(a.n_obs)) for g, j in col.items()}
    ng = np.asarray((X > 0).sum(1)).ravel(); tot = np.asarray(X.sum(1)).ravel()
    da = (gx["TH"] > 0) & ((gx["SLC6A3"] > 0) | (gx["SLC18A2"] > 0)) & (ng >= 500)
    donor_col = next((c for c in ["donor_id", "donor", "Donor"] if c in a.obs), None)
    obs = a.obs.copy(); obs["DA"] = da; obs["AGTR1pos"] = da & (gx["AGTR1"] > 0)
    obs["CALB1pos"] = da & (gx["CALB1"] > 0); obs["SOX6pos"] = da & (gx["SOX6"] > 0)
    obs["genes"] = ng; obs["umis"] = tot
    by = obs.groupby(donor_col)
    s = pd.DataFrame({"nuclei": by.size(), "median_genes": by["genes"].median(), "median_UMIs": by["umis"].median(),
                      "DA_candidates": by["DA"].sum(), "DA_SOX6pos": by["SOX6pos"].sum(),
                      "DA_AGTR1pos": by["AGTR1pos"].sum(), "DA_CALB1pos": by["CALB1pos"].sum()})
    for c in ["development_stage", "sex", "disease"]:
        if c in obs:
            s[c] = by[c].first().astype(str)
    s["dataset"] = r["title"]
    summary.append(s.reset_index().rename(columns={donor_col: "donor"}))
    sub = ad.AnnData(X[da].astype(np.float32), obs=obs.loc[da, [c for c in [donor_col, "development_stage", "sex", "disease",
                     "supercluster_term", "cluster_id", "subcluster_id", "cell_type", "dissection"] if c in obs]].copy(),
                     var=pd.DataFrame(index=pd.Index(sym).astype(str)))
    sub.var_names_make_unique()
    sub.obs["total_counts_raw"] = tot[da]; sub.obs["n_genes_raw"] = ng[da]
    sub.obs["donor"] = obs.loc[da, donor_col].astype(str).to_numpy()
    sub.write_h5ad(f"{OUT}/da_candidates/{r['dataset_id']}_DA.h5ad", compression="gzip")
    del a, X
summ = pd.concat(summary)
summ.to_csv(f"{OUT}/tables/DA_counts_per_donor.csv", index=False)
print("\nDopamine-neuron candidates per donor (TH plus SLC6A3 or SLC18A2, >= 500 genes):")
print(summ.to_string(index=False))
